## Module 2 Class activities
This notebook is a starting point for the exercises and activities that we'll do in class.

Before you attempt any of these activities, make sure to watch the video lectures for this module.

### Scraping permit data
Here's the code that we saw in the video lecture that queries the City of Seattle permit website, gets a dataframe of permits (including the URL), and then digs down further into that permit-specific URL.

In [2]:
# get the permit data from the API
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = 'https://data.seattle.gov/resource/ht3q-kdvx.json' # copied and pasted from the webpage
r = requests.get(url)
df = pd.DataFrame(json.loads(r.text))

df = df.head(8) # get the first 5 rows, so we don't overload the city's website.

# get an example link
permiturl = df.loc[7,'link']['url']
print(permiturl)

# request that page and get the soup object
r = requests.get(permiturl)
soup = BeautifulSoup(r.text)
print(soup.prettify())

https://services.seattle.gov/portal/customize/LinkToRecord.aspx?altId=3001610-LU
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html lang="en-US" ng-app="appAca" xml:lang="en-US" xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://ogp.me/ns#">
 <head id="ctl00_Head1">
  <link href="../App_Themes/Default/form.css" rel="stylesheet" type="text/css"/>
  <link href="../App_Themes/Default/style.css" rel="stylesheet" type="text/css"/>
  <title>
   Accela Citizen Access
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <style type="text/css">
   body, html {
        overflow-y: visible!important;
        }
        .page-overlay-blocker {
          position: fixed;
          z-index: 999;
          top: 0;
          left: 0;
          right: 0;
          bottom: 0;
  

In [81]:
# then we wrote this code to extract the project description 
links = soup.find_all('td')
for link in links:
    if 'Project Description' in link.text: 
        sublinks = link.find_all('td')
        description = sublinks[1].text
        # once we find a description, we exit
        break
    
print(description)

Land Use Application to subdivide one development site into six unit lots. The construction of residential units has been approved under Project #2503285/6100827. This subdivision of property is only for the purpose of allowing sale or lease of the unit lots.  Development standards will be applied to the original parcel and not to each of the new unit lots.


<div class="alert alert-block alert-info">
<strong>Exercise:</strong> If you look at the example, there may be a section giving information on the number of curb cuts. Extract that to a variable and print it.
</div>

In [9]:
# Hints
# Not all of the records have the curb cut field. You can see an example here:
# https://services.seattle.gov/portal/customize/LinkToRecord.aspx?altId=3001776-LU

# If you look at that webpage, you'll see that the text "Number of Curb Cuts for This Permit: "
# is within "span" tags
curbcuttext = soup.find("span", string="Number of Curb Cuts for This Permit: ")

# So to get the number of curb cuts (which is the next piece of text), 
# you can ask for the NEXT tag using find_next()

#.find_next() は BeautifulSoup における便利なメソッドの1つで、HTMLの中で「現在の要素のすぐ後ろにある要素を探す」ときに使う
#
n_curbcuts = curbcuttext.find_next()

# you'll need to add an if statement to deal with the case when this text does not exist

# your code here

AttributeError: 'NoneType' object has no attribute 'find_next'

In [82]:
curbcuttext = soup.find("span", string="Number of Curb Cuts for This Permit: ")

In [83]:
print(curbcuttext)

None


In [84]:
import numpy as np

if curbcuttext is None:
    n_curbcuts = np.nan
else:
    n_curbcuts = curbcuttext.find_next()

print(n_curbcuts)

nan


<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Now turn that into a function that you can apply to each row of your dataframe. Add a new column, <strong>n_curbcuts</strong>, to your dataframe.
</div>

In [85]:
df.columns

Index(['permitnum', 'permitclass', 'permitclassmapped', 'permittypemapped',
       'description', 'statuscurrent', 'originaladdress1', 'originalcity',
       'originalstate', 'originalzip', 'link', 'latitude', 'longitude',
       'location1', 'housingunitsremoved', 'housingunitsadded', 'applieddate',
       'issueddate', 'expiresdate', 'decisiondate', 'permittypedesc',
       'contractorcompanyname', 'estprojectcost'],
      dtype='object')

In [86]:
print(df['link'])

0    {'url': 'https://services.seattle.gov/portal/c...
1    {'url': 'https://services.seattle.gov/portal/c...
2    {'url': 'https://services.seattle.gov/portal/c...
3    {'url': 'https://services.seattle.gov/portal/c...
4    {'url': 'https://services.seattle.gov/portal/c...
5    {'url': 'https://services.seattle.gov/portal/c...
6    {'url': 'https://services.seattle.gov/portal/c...
7    {'url': 'https://services.seattle.gov/portal/c...
Name: link, dtype: object


In [87]:
def curbcuts(x):
    permiturl = x['url']
    r = requests.get(permiturl)
    soup = BeautifulSoup(r.text)
    
    curbcuttext = soup.find("span", string="Number of Curb Cuts for This Permit: ")
    if curbcuttext is None:
        n_curbcuts = np.nan
    else:
        n_curbcuts = int(curbcuttext.find_next().text)
            
df['n_curbcuts'] = df['link'].apply(curbcuts)

In [88]:
df.head(8)

,permitnum,permitclass,permitclassmapped,permittypemapped,description,statuscurrent,originaladdress1,originalcity,originalstate,originalzip,...,housingunitsremoved,housingunitsadded,applieddate,issueddate,expiresdate,decisiondate,permittypedesc,contractorcompanyname,estprojectcost,n_curbcuts
0,3001212-LU,Single Family/Duplex,Residential,Master Use Permit,PROJECT CANCELLED 12/8/2010 -- This short plat...,Canceled,6519 S BANGOR ST,SEATTLE,WA,98178,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3001271-LU,Single Family/Duplex,Residential,Master Use Permit,Land Use Permit to adjust the boundary between...,Completed,4226 1ST AVE NW,SEATTLE,WA,98107,...,0.0,0.0,2005-12-16,2006-05-15,2007-11-15,2006-05-10,NaN,NaN,NaN,None
2,3001310-LU,Single Family/Duplex,Residential,Master Use Permit,Land use application to adjust the boundary be...,Completed,941 23RD AVE S,SEATTLE,WA,98144,...,NaN,NaN,2007-02-14,2008-08-28,2011-08-14,2008-08-13,NaN,NaN,NaN,None
3,3001312-LU,N/A,N/A,Master Use Permit,Cancelled due to no activity for more than 9 y...,Canceled,3131 E MADISON ST,SEATTLE,WA,98112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3001440-LU,Commercial,Non-Residential,Master Use Permit,PROJECT CANCELLED 5/23/2011 -- Project On Hold...,Canceled,9030 13TH AVE NW,SEATTLE,WA,98117,...,NaN,NaN,2005-08-12,NaN,NaN,NaN,NaN,NaN,NaN,None
5,3001442-LU,Single Family/Duplex,Residential,Master Use Permit,Land Use Permit to subdivde two parcels into t...,Completed,7960 46TH AVE S,SEATTLE,WA,98118,...,0.0,0.0,2005-10-26,2007-04-23,2009-08-24,2006-04-27,NaN,NaN,NaN,None
6,3001452-LU,Multifamily,Residential,Master Use Permit,Land use permit to subdivide 1 parcel into 6 u...,Completed,4017 SW ADMIRAL WAY,SEATTLE,WA,NaN,...,NaN,NaN,2005-12-02,2006-05-26,2007-11-26,2006-04-04,NaN,NaN,NaN,None
7,3001610-LU,Multifamily,Residential,Master Use Permit,Land Use Application to subdivide one developm...,Completed,918 2ND AVE W,SEATTLE,WA,98119,...,NaN,NaN,2005-12-14,2012-07-12,2015-02-28,2012-02-15,NaN,NaN,NaN,None


In [ ]:


url = 'https://data.seattle.gov/resource/ht3q-kdvx.json' # copied and pasted from the webpage
r = requests.get(url)
df = pd.DataFrame(json.loads(r.text))

df = df.head(8) # get the first 5 rows, so we don't overload the city's website.

# get an example link
permiturl = df.loc[7,'link']['url']
print(permiturl)

# request that page and get the soup object
r = requests.get(permiturl)
soup = BeautifulSoup(r.text)
print(soup.prettify())

In [10]:
# your code here

def curbcut_count(soup):
    curbcuttext = soup.find("span", string="Number of Curb Cuts for This Permit: ")

    if curbcuttext is not None:
        n_curbcuts_tag = curbcuttext.find_next()
        if n_curbcuts_tag is not None:
            n_curbcuts = n_curbcuts_tag.text
        else:
            n_curbcuts = None
    else:
        n_curbcuts = None

df['n_curbcuts'] = df.apply(curbcut_count)

AttributeError: 'Series' object has no attribute 'find'

### Fixing errors
We'll do more scraping in just a moment. But first, let's do some examples of how to interpret an error message, and fix it.

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Each of the cells below will generate an error. Look at the error message and see if you can figure out how to fix it. (Don't Google it until you try to figure it out based on the error message.)
</div>

In [89]:
print(df.housingunitsadded)
print(df.housingunitsremoved)

0    NaN
1    0.0
2    NaN
3    NaN
4    NaN
5    0.0
6    NaN
7    NaN
Name: housingunitsadded, dtype: object
0    NaN
1    0.0
2    NaN
3    NaN
4    NaN
5    0.0
6    NaN
7    NaN
Name: housingunitsremoved, dtype: object


In [92]:
#もともとNaNを含むstrで、intに変えてもいいかと思ったけど、intはNaNを許容できない
#一方でfloatはNaNを許容できる
df['housingunitsadded'] = df.housingunitsadded.astype(float)
df['housingunitsremoved'] = df.housingunitsremoved.astype(float)

In [95]:
# the housingunitsremoved and housingunitsadded give useful information
# let's create a new column with netunits
df['netunits'] = df.housingunitsadded - df.housingunitsremoved
print(df.netunits)

0    NaN
1    0.0
2    NaN
3    NaN
4    NaN
5    0.0
6    NaN
7    NaN
Name: netunits, dtype: float64


In [102]:
df[['housingunitsadded','housingunitsremoved','netunits']].head(50)
#なぜここで[]を2つ用意しないといけない？

#[]を2回使うのは、「列名リストを渡すため」に必要だから
#外側の [] は「列を指定する操作」、内側の [] は 列名のリスト（複数形）
#df['netunits']だと、Series（1次元） だが、
#df[['housingunitsadded', 'housingunitsremoved', 'netunits']]はDataFrame（2次元） 

,housingunitsadded,housingunitsremoved,netunits
0,NaN,NaN,NaN
1,0.0,0.0,0.0
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,0.0,0.0,0.0
6,NaN,NaN,NaN
7,NaN,NaN,NaN


In [97]:
# original prompt
# print the address of the first row
print('Address of first row is {}. Permit type is {}'.format(df.iloc[0].originaladdress1))

IndexError: Replacement index 1 out of range for positional args tuple

In [98]:
# print the address of the first row
print('Address of first row is {}. Permit type is {}'.format(df.iloc[0].originaladdress1, df.iloc[0].permittypemapped))

Address of first row is 6519 S BANGOR ST. Permit type is Master Use Permit


In [26]:
# original
# Convert the number of housing units to integers
# and then summarize

df['unitsadded_numeric'] = df.housingunitsadded.astype(int)
df.unitsadded_numeric.describe(

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [99]:
# original
# Convert the number of housing units to integers
# and then summarize

df['unitsadded_numeric'] = df.housingunitsadded.astype(float)
df.unitsadded_numeric.describe()

count    2.0
mean     0.0
std      0.0
min      0.0
25%      0.0
50%      0.0
75%      0.0
max      0.0
Name: unitsadded_numeric, dtype: float64

### Scraping craigslist

In the lecture, we saw how to scrape the main page (the list of posts).

What if you want to get more information about (say) a particular apartment?

Go to the [craigslist housing page](https://losangeles.craigslist.org/search/apa#search=1~gallery~0~0) and copy the link for one of the listings. It should look something like this:
https://losangeles.craigslist.org/lgb/apa/d/long-beach-home-for-rent/7597309102.html

(It's fine to copy and paste the URL for now. A second step would be to loop over the URLs from the dataframe of postings that we created in the video lecture, but in class, we'll just focus on one example.)

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> For this URL, use requests to get the content of the post. (No need to create a soup object yet.)
</div>

In [104]:
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

# your code here
# put the output of the request in a variable called r
url = 'https://losangeles.craigslist.org/sfv/apa/d/tarzana-sparkling-pool-gated-community/7833430617.html' # copied and pasted from the webpage
r = requests.get(url)

# so you can access the content like this
print(r.content)

b'<!DOCTYPE html>\n<html>\n<head>\n    \n\t<meta charset="UTF-8">\n\t<meta http-equiv="X-UA-Compatible" content="IE=Edge">\n\t<meta name="viewport" content="width=device-width,initial-scale=1">\n\t<meta property="og:site_name" content="craigslist">\n\t<meta name="twitter:card" content="preview">\n\t<meta property="og:title" content="Sparkling Pool, Gated Community, Internet Access - apts/housing for rent - apartment rent - craigslist">\n\t<meta name="description" content="5825 Reseda Blvd, Tarzana, CA 91356 With its lush landscape, Tarzana Springs is the home you&#39;ve been looking for. Our beautiful one- and two-bedroom units are perfect for you and your family, with...">\n\t<meta property="og:description" content="5825 Reseda Blvd, Tarzana, CA 91356 With its lush landscape, Tarzana Springs is the home you&#39;ve been looking for. Our beautiful one- and two-bedroom units are perfect for you and your family, with...">\n\t<meta property="og:image" content="https://images.craigslist.org

Now let's extract more information from the page. We have a couple of strategies here. First, we could skip trying to parse the page with `BeautifulSoup`, and just see if particular bits of text are present.

For example, what transportation modes does the post emphasize? Do they mention Section 8 vouchers? Some of this might be exploratory—we can see what type of language is included, and then parse in a more structured way (e.g. distinguishing between "No Section 8" and "Section 8 welcome").

<div class="alert alert-block alert-info">
    <strong>Exercise:</strong> Write a function that will return True if Section 8 is mentioned, otherwise False.

*Hint*: the `in` operator is a simple way to do this. For example:

In [105]:
'plan' in 'urban planning'

True

In [106]:
'plan' in 'Urban Planning' 

False

In [107]:
import os
print(os.getcwd())

/Users/Tomo/Desktop/urbandatascience/classes_tomo


In [121]:
def checkSec8(url):
    r = requests.get(url)
    if "section 8" in r.text:
        return True
    else:
        return False

checkSec8(url)

False

In [108]:
#'data/cl_posts_ver1.html' というファイルを 読み込み専用（'r'） で開く
#with 文は、ファイルを開いて使い終わったあとに自動的に閉じてくれる安全な書き方
#fは開いたファイルを表すオブジェクト（ハンドル）。他の名前でもOK
with open('data/cl_posts_ver1.html', 'r') as f:

    #ファイルの中身をすべて読み取って、saved_content という変数に保存（中身はHTMLの文字列）
    saved_content = f.read() 

#BeautifulSoupライブラリを使ってHTMLの文字列（saved_content）を解析
#html.parser はPythonに内蔵されたHTMLパーサー
#HTMLパーサー（HTML parser）とは、HTML文書を「意味のある構造」として理解・処理できる形に変換するプログラムのこと
soup = BeautifulSoup(saved_content, features='html.parser')

#HTMLを インデント付きの見やすい形式 に整形して出力
print(soup.prettify())

<!DOCTYPE html>
<!-- saved from url=(0098)https://losangeles.craigslist.org/sfv/apa/d/tarzana-sparkling-pool-gated-community/7833430617.html -->
<html class="cl-not-small-viewport cl-landscape js canvas draggable fileAPI geolocation matchMedia picture pushState placeholder no-touchCapable transitions">
 <head>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="craigslist" property="og:site_name"/>
  <meta content="preview" name="twitter:card"/>
  <meta content="Sparkling Pool, Gated Community, Internet Access - apts/housing for rent - apartment rent - craigslist" property="og:title"/>
  <meta content="5825 Reseda Blvd, Tarzana, CA 91356 With its lush landscape, Tarzana Springs is the home you've been looking for. Our beautiful one- and two-bedroom units are perfect for you and your family, with..." name="description"/>
  <m

In [109]:
# your code here to return Section 8 information

'Section 8' in soup

False

In [3]:
#r.textを使う場合と、BeautifulSoupを使う場合の違いとは？

#r.textを使う方法は、HTML全体を文字列として取得している。
#なので、特定の単語がページに含まれているかだけを見たいときに便利（構造は無視）。
#スクレイピングの前段階でのフィルター処理などに向いている。
#BeautifulSoupを使うと、構造を解析して検索しやすくなる。
#つまり、r.textは構造を無視してしまうが、BeautifulSoupはposts = soup.find_all('li', class_='cl-search-result’)のようにレイヤーを指定しての検索ができる。

#BeautifulSoupは、htmlファイルをダウンロードせずとも、下の構文のようにrequestsを使ってオンライン上の内容に直接使える
#lecture2cでBeautifulSoupをダウンロードデータに使っていたのは、Craigslistがrequestsを受け付けなくなっているから

url = 'https://losangeles.craigslist.org/sfv/apa/d/tarzana-sparkling-pool-gated-community/7833430617.html' # copied and pasted from the webpage
r = requests.get(url)
soup = BeautifulSoup(r.text, features='html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="craigslist" property="og:site_name"/>
  <meta content="preview" name="twitter:card"/>
  <meta content="Sparkling Pool, Gated Community, Internet Access - apts/housing for rent - apartment rent - craigslist" property="og:title"/>
  <meta content="5825 Reseda Blvd, Tarzana, CA 91356 With its lush landscape, Tarzana Springs is the home you've been looking for. Our beautiful one- and two-bedroom units are perfect for you and your family, with..." name="description"/>
  <meta content="5825 Reseda Blvd, Tarzana, CA 91356 With its lush landscape, Tarzana Springs is the home you've been looking for. Our beautiful one- and two-bedroom units are perfect for you and your family, with..." property="og:description"/>
  <meta content="https://images.craigslist.org/00X0X_aOCL2zverPC_0uY0hq_600x450.jpg"

Most of the post is free-form text. So there's not going to be much value added by `BeautifulSoup`.

The exceptions are (i) parking, and (ii) the geographic coordinates.

<div class="alert alert-block alert-info">
    <strong>Exercise:</strong> Write a function that will return True if the apartment has no parking, and also returns the lat/lon of the apartment

*Hint*: First, create a `soup` object. Then, look and see what tag and class encloses this information. Then, you can experiment with `find` and `find_all` with this tag and class.

In [111]:
# your code here

parking = soup.find_all('div', class_='attrgroup')

In [112]:
print(parking)

[<div class="attrgroup">
<span class="attr important">
                1BR / 1Ba
            </span>
<span class="attr important">
                770ft<sup>2</sup>
</span>
<span class="attr important available-now">available now</span>
</div>, <div class="attrgroup">
<div class="attr rent_period">
<span class="labl">rent period:</span>
<span class="valu"> <a href="https://losangeles.craigslist.org/search/apa?rent_period=3">monthly</a>
</span>
</div>
</div>, <div class="attrgroup">
<div class="attr pets_cat">
<span class="valu"> <a href="https://losangeles.craigslist.org/search/apa?pets_cat=1">cats are OK - purrr</a>
</span>
</div>
<div class="attr">
<span class="valu"> <a href="https://losangeles.craigslist.org/search/apa?housing_type=1">apartment</a>
</span>
</div>
<div class="attr pets_dog">
<span class="valu"> <a href="https://losangeles.craigslist.org/search/apa?pets_dog=1">dogs are OK - wooof</a>
</span>
</div>
<div class="attr">
<span class="valu"> <a href="https://losangeles.cr

Now you've written this code, a next step would be to package it in a function that you can apply to all the URLs in your dataframe of posts (like the one we created in the video lecture). 

## Large language models [Optional]

Large language models (LLMs) such as ChatGPT can also be accessed via an API.

The APIs are changing very rapidly, as are the pricing structures. For now, some LLMs are offered for free, for limited use. One such model with a free tier is Gemini, by Google.

If you'd like to experiment with Gemini, you need to [get an API key here](https://aistudio.google.com/u/1/apikey). No credit card is necessary, but your UCLA Google account won't work - you need a personal Google account.

[The documentation and some examples are here](https://github.com/googleapis/python-genai).

The other challenge with Gemini is that its Python library has several incompatibilities, including with some of the ones we use elsewhere in the course. So you will need to create a new environment in Anaconda, using the same setup process (importing an environment) as you did at the [start of the course](https://urbandatascience.its.ucla.edu/getting-started/).

Specifically:
- In Anaconda, go to the Environments tab
- Click on Import
- Choose `google-genai-env.yml` under Local Drive, and call the environment `genai`
- Manually add the `google-genai` package (choose "Not Installed", type `google-genai` into the search bar, select the checkbox, and then click Apply at the bottom of the screen)
- Anaconda will take some time, before you can click Apply again to install
- Close this notebook and open it again after you switch to your new `genai` environment

All set up? Let's look at a simple example---passing a query to the chat interface.

In [ ]:
from google import genai  # if this doesn't load, you probably have the wrong environment
gemini_api_key = 'XXXX' #'XXXXX' # fill in your key here

c = genai.Client(api_key=gemini_api_key)
chat = c.chats.create(model='gemini-2.0-flash-001')
response = chat.send_message('What do urban planners need to learn about gen AI?')
print(response.text)

How might this be used in web scraping? 

Well, perhaps you can ask it to parse the text.

<div class="alert alert-block alert-info">
    <strong>Exercise:</strong> Pass the text of the craiglist post to Gemini, and ask it to return the number of parking spaces (if any). Add the result to your dataframe. NOTE: This exercise is optional (you might not want to create a Google account to get an API key).

In [ ]:
# your code here

We won't do it here, but [you can also experiment with the MetaAI API](https://github.com/Strvm/meta-ai-api). The advantage: it doesn't need an API key, but the capabilities are a bit more limited. You can also find the Meta installed in the `genai` environment.

<div class="alert alert-block alert-info">
<h3>What you should have learned</h3>
<ul>
  <li>Gain confidence in experimenting with code - exploring different objects, writing functions, and so on</li>
  <li>Learn how to extract information from a scraped webpage - how to do the detective work.</li>
  <li>Gain confidence in debugging errors.</li>
  <li>Learn how to integrate Large Language Models into Python</li>
</ul>
</div>